In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import torch

In [17]:
data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/mme/mme_base_des_05_11_2025.csv")

In [18]:
data.head()

,question,gt_answer,question_id,image_id,image_path,data_type,answer
0,Is there a blue court in the image? Please ans...,Yes,231eab02-8ebe-4b46-b47e-4ae24c83b59e,12120,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"Yes, there is a blue court in the image."
1,Is there a purple court in the image? Please a...,No,c7840ca0-1b16-4c44-8b86-a7c95b5ba0aa,12120,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"Yes, there is a purple court in the image."
2,Is there a red couch in the image? Please answ...,Yes,4d5a9202-def3-4fdc-a5a4-cdd2578f98a1,564280,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"Yes, there is a red couch in the image."
3,Is there a black couch in the image? Please an...,No,5d454889-4297-4834-8c81-9fa1f04dc95b,564280,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"No, there is no black couch in the image. The ..."
4,Is there a white plate in the image? Please an...,Yes,5969601a-b454-4085-b0a0-e760126a0414,8277,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/mme...,color,"Yes, there is a white plate in the image."


In [19]:
preds = []
for i in data["answer"]:
    pred = i[:10].lower()
    if "yes" in pred:
        preds.append("Yes")
    elif "no" in pred:
        preds.append("No")
    else:
        preds.append("unknown")

In [20]:
pd.Series(preds).value_counts()

Yes    128
No     112
Name: count, dtype: int64

In [23]:
accuracy_score(data["gt_answer"], preds)

0.8666666666666667

In [24]:
f1_score(data["gt_answer"], preds, average='weighted')

0.8665183537263625

# tatget-word based evaluation

In [9]:
result_df = pd.read_pickle("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/mme/combined_stage_label_with_evidence_and_mlp_06_11_2025.pkl")

In [10]:
all_preds = []
for ans in result_df["answer"]:
    pred = ans[:10].lower()
    if "yes" in pred:
        all_preds.append("Yes")
    elif "no" in pred:
        all_preds.append("No")
    else:
        all_preds.append("Unknown")

In [11]:
result_df["pred_label"] = all_preds

In [12]:
result_df["pred_label"].value_counts()

pred_label
Yes    128
No     112
Name: count, dtype: int64

In [ ]:
all_new_labels = []
for inx, row in result_df.iterrows():
    old_label = row["pred_label"]
    pred = row["labels_with_evidence"][0]["label"]
    if pred >= 0.5:
        new_label = old_label
    else:
        if old_label == "Yes":
            new_label = "No"
        elif old_label == "No":
            new_label = "Yes"
        else:
            new_label = "unknown"
    all_new_labels.append(new_label)
    

In [55]:
faild = []
corrrected_label = []
for inx, row in result_df.iterrows():
    old_label = row["pred_label"]
    try:
        prob = row["labels_with_evidence"][0]["label"]
        img_evi = (torch.tensor(row["labels_with_evidence"][0]["evidence"]) >= 0.45).int().sum().item()
        if prob >= 0.5:
            new_label = old_label
        elif img_evi >= 5 and prob < 0.95:
            new_label = old_label
        else:
            if old_label == "Yes":
                new_label = "No"
            elif old_label == "No":
                new_label = "Yes"
            else:
                new_label = "unknown"
        corrrected_label.append(new_label)

    except Exception as e:
        print(e)
        faild.append(inx)
        corrrected_label.append(old_label)

In [56]:
result_df["corrected_label"] = corrrected_label

In [57]:

for name, df in result_df.groupby("data_type"):
    acc_score = accuracy_score(df["gt_answer"], df["corrected_label"])
    df.index = pd.RangeIndex(start=0, stop=len(df), step=1)
    s = (df["gt_answer"] == df["corrected_label"])
    if len(s) % 2 != 0:
        s = s[:-1]
    result = s.values.reshape(-1, 2).all(axis=1)
    result_series = pd.Series(result)
    acc_pp =  result_series.value_counts().to_dict()[True]/result_series.shape[0]
    final_score = (acc_score + acc_pp)*100
    print(name)
    print("final score: {}".format(final_score))
    print("---------------")   

color
final score: 160.0
---------------
count
final score: 163.33333333333334
---------------
existence
final score: 195.0
---------------
position
final score: 128.33333333333331
---------------


# evidence conditioned detection

In [139]:
import torch

faild = []
corrrected_label = []
for inx, row in result_df.iterrows():
    old_label = row["pred_label"]
    try:
        prob = row["labels_with_evidence"][0]["label"]
        img_evi = (torch.tensor(row["labels_with_evidence"][0]["evidence"]) >= 0.4).int().sum().item()
        if prob >= 0.6:
            new_label = old_label
        elif img_evi >= 2 and prob < 0.6:
            new_label = old_label
        else:
            if old_label == "Yes":
                new_label = "No"
            elif old_label == "No":
                new_label = "Yes"
            else:
                new_label = "unknown"
        corrrected_label.append(new_label)

    except Exception as e:
        print(e)
        faild.append(inx)
        corrrected_label.append(old_label)

In [140]:
result_df["corrected_label"] = corrrected_label

In [142]:

for name, df in result_df.groupby("data_type"):
    acc_score = accuracy_score(df["gt_answer"], df["corrected_label"])
    df.index = pd.RangeIndex(start=0, stop=len(df), step=1)
    s = (df["gt_answer"] == df["corrected_label"])
    if len(s) % 2 != 0:
        s = s[:-1]
    result = s.values.reshape(-1, 2).all(axis=1)
    result_series = pd.Series(result)
    acc_pp =  result_series.value_counts().to_dict()[True]/result_series.shape[0]
    final_score = (acc_score + acc_pp)*100
    print(name)
    print("final score: {}".format(final_score))
    print("---------------")   

color
final score: 160.0
---------------
count
final score: 163.33333333333334
---------------
existence
final score: 195.0
---------------
position
final score: 128.33333333333331
---------------
